# Random Forest Classifier on SMILES ChemNet Embeddings

### Basic Preprocessing

In [ ]:
# Import packages
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import functions_enc as f

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

In [ ]:
# The 5/20 dataset with rat based toxicity data
df2 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data2.csv")
print(df2.shape)
df2.head() 

In [ ]:
# Uniformity of ionization model labels
print(df2["Ionization_Mode"].unique())
df2["Ionization_Mode"] = df2["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df2["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df2 = df2[df2["Ionization_Mode"] != "'N/A'"]
print(df2["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df2
df2 = df2.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df2["SMILES_spectra"] = df2["SMILES_spectra"].str.replace("'", "")
df2.head()

In [ ]:
# Count unique SMILES in the DataFrame
unique_smiles_count = df2["SMILES_spectra"].nunique()
# Count how many SMILES have more than 20, 10, and 5 corresponding rows
smiles_counts = df2["SMILES_spectra"].value_counts()
more_than_20 = (smiles_counts > 20).sum()
more_than_10 = (smiles_counts > 10).sum()
more_than_5 = (smiles_counts > 5).sum()
more_than_1 = (smiles_counts > 1).sum()
just_1 = (smiles_counts == 1).sum()

# Print the results
print("Number of unique SMILES:", unique_smiles_count)
print("SMILES with >20 rows:", more_than_20)
print("SMILES with >10 rows:", more_than_10)
print("SMILES with >5 rows:", more_than_5)
print("SMILES with >1 rows:", more_than_1)
print("SMILES with 1 row:", just_1)



### Save ChemNet Dataframe for 5/20 dataset

In [ ]:
# Cate's smiles to ChemNet embedding code
def get_chemnet_emb_from_smiles(smiles_list):
    """
    Get ChemNet embeddings from a list of SMILES strings.

    Parameters:
    smiles_list (list): List of SMILES strings.

    Returns:
    dict: A dictionary mapping each SMILES string to its corresponding ChemNet embedding.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fcd = FCD(device, n_jobs=1)
    
    smiles_emb_dict = {}

    for smiles in smiles_list:
        try:
            emb = fcd.get_predictions([smiles])[0]
            smiles_emb_dict[smiles] = list(emb)
        except KeyError as e:
            if e == 'PropertyTable':
                smiles_emb_dict[smiles] = 'unknown'

    return smiles_emb_dict

In [ ]:
# Get the ChemNet embeddings for the SMILES in df2
ChemNet_df2_dict = get_chemnet_emb_from_smiles(df2["SMILES_spectra"].tolist())


In [ ]:
# Convert the dictionary to a DataFrame
ChemNet_df2 = pd.DataFrame.from_dict(ChemNet_df2_dict, orient='index')
# Name the first column to "SMILES" and check the dataframe
ChemNet_df2.reset_index(inplace=True)
ChemNet_df2.rename(columns={'index': 'SMILES'}, inplace=True)
ChemNet_df2.head()

In [ ]:
# Save the ChemNet embeddings to a CSV file
# ChemNet_df2.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_df2.csv", index=False)

In [ ]:
# Read in the csv file with the ChemNet embeddings
ChemNet_df2 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_df2.csv")
ChemNet_df2.head()

In [ ]:
df2.head(10)

In [ ]:
# Append the response column to the ChemNet DataFrame
# First remove all redundant SMILES from df2
df2_unique_smiles = df2.drop_duplicates(subset=["SMILES_spectra"])
# Merge Response from df2_unique_smiles onto ChemNet_df2 by matching SMILES columns
ChemNet_df2_withtox = ChemNet_df2.merge(
    df2_unique_smiles[['SMILES_spectra', 'Response']],
    left_on='SMILES',
    right_on='SMILES_spectra',
    how='left'
)

# Optionally, drop the now-redundant 'SMILES_spectra' column
ChemNet_df2_withtox.drop(columns=['SMILES_spectra'], inplace=True)


In [ ]:
print(ChemNet_df2_withtox.shape)
ChemNet_df2_withtox.head()

### One hot encode toxicity

The EPA toxicity criteria consist of 4 levels:
1. Up to and includeing 50 mg/kg
2. From 50 through 500 mg/kg
3. From 500 though 5000 mg/kg
4. Greater than 5000 mg/kg

In [ ]:
# Define a function to assign EPA levels
def assign_epa_level(response):
    if response <= 50:
        return "EPA_level_1"
    elif response <= 500:
        return "EPA_level_2"
    elif response <= 5000:
        return "EPA_level_3"
    else:
        return "EPA_level_4"



In [ ]:
# Initialize a new DataFrame for the EPA levels
ChemNet_df2_epalevels = ChemNet_df2_withtox.copy()

# Assign EPA levels
ChemNet_df2_epalevels["EPA_level"] = ChemNet_df2_withtox["Response"].apply(assign_epa_level)

# One hot encode the EPA_level column
ChemNet_df2_epalevels = pd.get_dummies(ChemNet_df2_epalevels, columns=["EPA_level"], prefix='',prefix_sep='')

# Convert boolean columns to int (1/0)
epa_cols = [col for col in ChemNet_df2_epalevels.columns if col.startswith("EPA_level_")]
ChemNet_df2_epalevels[epa_cols] = ChemNet_df2_epalevels[epa_cols].astype(int)

# Remove the Response column
ChemNet_df2_epalevels.drop(columns=["Response"], inplace=True)

# Check the shape and head of the DataFrame
print(ChemNet_df2_epalevels.shape)
ChemNet_df2_epalevels.head()

In [ ]:
# Lets get counfst of each EPA level
# Count the number of entries at each EPA_level
epa_level_cols = [col for col in ChemNet_df2_epalevels.columns if col.startswith("EPA_level_")]
epa_level_counts = ChemNet_df2_epalevels[epa_level_cols].sum()
print(epa_level_counts)

### Train a Random Forest Classifier

In [ ]:
# Load the needed packages
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Prepare features (X) and labels (y)
# Drop SMILES and keep only embedding columns for X, and EPA levels for y
embedding_cols = [col for col in ChemNet_df2_epalevels.columns if col not in ['SMILES'] + [col for col in ChemNet_df2_epalevels.columns if col.startswith('EPA_level_')]]
epa_level_cols = [col for col in ChemNet_df2_epalevels.columns if col.startswith('EPA_level_')]

X = ChemNet_df2_epalevels[embedding_cols]
y = ChemNet_df2_epalevels[epa_level_cols].idxmax(axis=1)  # Get the EPA level label

In [ ]:
X.head()

In [ ]:
print(y)

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=None, max_features='sqrt', random_state=47)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)

In [ ]:
# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]

# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test, y_pred, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Balances 4 way Random Forest Classifier
# Set the target number of samples per class
target_count = 35
class_counts = dict(Counter(y))
sampling_strategy = {cls: min(target_count, class_counts[cls]) for cls in class_counts}
# First undersample to at most 30 per class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=15)
X_under, y_under = rus.fit_resample(X, y)

# Now oversample to exactly 30 per class
sampling_strategy = {cls: target_count for cls in class_counts}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=15)
X_balanced, y_balanced = ros.fit_resample(X_under, y_under)

# Train/test split
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=15, stratify=y_balanced
)

# Train Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=42)
rf_bal.fit(X_train_bal, y_train_bal)

# Predict and evaluate
y_pred_bal = rf_bal.predict(X_test_bal)
#print(confusion_matrix(y_test_bal, y_pred_bal))

In [ ]:
# Print and plot confusion matrix in numerical order
cm = confusion_matrix(y_test_bal, y_pred_bal, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Define parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2']
}

# Initialize Random Forest
rf = RandomForestClassifier(random_state=47)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', # Use macro F1 score for multi-class classification
    n_jobs=-1,
    verbose=2
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Use the best estimator to predict and evaluate
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# Define the EPA levels in numerical order
epa_order = ["EPA_level_1", "EPA_level_2", "EPA_level_3", "EPA_level_4"]

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=epa_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=epa_order)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Tuned RF)")
plt.show()


In [ ]:
# Print best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

Do the hyperparameter search with F1 score due to imbalance in the data causign inflated accuracy. 
Consider 1 union 2 


### Binary classifcation problem

We can make this a binary classfication problem and mitigate some of the imbalance issues that we saw if we remove the EPA level 3 and combine levels 1 and 2. Is this a reasonable think to do form a chemistry standpoint?

In [ ]:
# Combine EPA_level_1 and EPA_level_2 into EPA_level_1_2, remove EPA_level_3, keep only EPA_level_1_2 and EPA_level_4

df = ChemNet_df2_epalevels.copy()

# Create new binary label
def combine_epa_levels(row):
    if row['EPA_level_1'] == 1 or row['EPA_level_2'] == 1:
        return 'EPA_level_1_2'
    elif row['EPA_level_4'] == 1:
        return 'EPA_level_4'
    else:
        return None  # EPA_level_3 or anything else

df['EPA_binary'] = df.apply(combine_epa_levels, axis=1)

# Filter out rows that are not EPA_level_1_2 or EPA_level_4
df_bin = df[df['EPA_binary'].isin(['EPA_level_1_2', 'EPA_level_4'])].copy()

# Prepare features and labels
embedding_cols = [col for col in df_bin.columns if col not in ['SMILES', 'EPA_binary'] + [col for col in df_bin.columns if col.startswith('EPA_level_')]]
X_bin = df_bin[embedding_cols]
y_bin = df_bin['EPA_binary']

# Train/test split
from sklearn.model_selection import train_test_split
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

# Set up Random Forest and GridSearchCV for binary classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2']
}

rf_bin = RandomForestClassifier(random_state=47)

grid_search_bin = GridSearchCV(
    estimator=rf_bin,
    param_grid=param_grid,
    cv=5,
    scoring='f1',  # binary F1
    n_jobs=-1,
    verbose=2
)

grid_search_bin.fit(X_train_bin, y_train_bin)

# Predict and evaluate
best_rf_bin = grid_search_bin.best_estimator_
y_pred_bin = best_rf_bin.predict(X_test_bin)


In [ ]:
# Confusion matrix for binary classification
cm_bin = confusion_matrix(y_test_bin, y_pred_bin, labels=['EPA_level_1_2', 'EPA_level_4'])
disp_bin = ConfusionMatrixDisplay(confusion_matrix=cm_bin, display_labels=['EPA_level_1_2', 'EPA_level_4'])
disp_bin.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Binary, Tuned RF)")
plt.show()

print("Best parameters:", grid_search_bin.best_params_)
print("Best cross-validation F1:", grid_search_bin.best_score_)